# Held-Out Inference Evaluation — Topology Reconfiguration (Case Study A)

Quantitative evaluation for the WAN topology-reconfiguration case study (paper Sec. V-A, Fig. 13),
addressing the reviewers' request for metrics instead of a single qualitative figure.

**Idea.** Instead of one hand-labelled `Link_Down(Berlin, Hamburg)` picture, we:

1. **Sweep** every link as the down element and **generate labels programmatically**.
2. Split each scenario into a **known** region (grounding facts fed into the LTN loss) and a
   **held-out** region (labels withheld). The model can only reach held-out labels by propagating
   the injected rules and by the predicate MLP generalising over the fixed coordinate embeddings.
3. Score the model's continuous truth values on the **held-out** entities only, against two
   independent ground-truth definitions, over multiple seeds -> report mean +/- std.

**Two ground-truth definitions** (see `gt_topological` / `gt_demand_footprint`):

* **Topological** — 1-hop failure spreading (the paper's assumption). Also used to build the
  *known* grounding facts, matching the case study.
* **Demand-based (reroute footprint)** — which links/nodes actually carry rerouted traffic after
  the cut, computed from the SNDlib demand matrix. This is **independent of the injected rules**,
  so recovering it is a non-circular test that the model's topological reasoning also predicts real
  traffic impact.

**Why this is not circular.** For any held-out entity, its label was never asserted in the loss.
Sanity check: every held-out label is computed by `gt_*` from the topology/demands, never read off
anything placed in the loss.

**Requirements:** `ltn` (LTNtorch), `torch`, `networkx`, `scikit-learn`, `pandas`, `numpy`.
Runs on GPU if available. Set `CONFIG['quick_test']=True` for a fast smoke run first.


In [ ]:
import os, random, itertools
import numpy as np
import torch
import networkx as nx
import pandas as pd
import ltn
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, accuracy_score

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print("device:", device)

In [ ]:
# ----------------------------- Topology (germany17 / SNDlib, ref [27]) -----------------------------
raw_nodes = {
    "Hannover": (9.80, 52.39), "Frankfurt": (8.66, 50.14), "Hamburg": (10.08, 53.55),
    "Norden": (7.21, 53.60), "Bremen": (8.80, 53.08), "Berlin": (13.48, 52.52),
    "Muenchen": (11.55, 48.15), "Ulm": (9.99, 48.40), "Nuernberg": (11.08, 49.45),
    "Stuttgart": (9.12, 48.73), "Karlsruhe": (8.41, 49.01), "Mannheim": (8.49, 49.49),
    "Essen": (7.00, 51.44), "Dortmund": (7.48, 51.51), "Duesseldorf": (6.78, 51.22),
    "Koeln": (7.01, 50.92), "Leipzig": (12.38, 51.34),
}
raw_links = {
    "L1": ("Berlin","Hamburg"), "L2": ("Berlin","Hannover"), "L3": ("Berlin","Leipzig"),
    "L4": ("Bremen","Hamburg"), "L5": ("Bremen","Hannover"), "L6": ("Bremen","Norden"),
    "L7": ("Dortmund","Essen"), "L8": ("Dortmund","Hannover"), "L9": ("Dortmund","Koeln"),
    "L10": ("Dortmund","Norden"), "L11": ("Duesseldorf","Essen"), "L12": ("Duesseldorf","Koeln"),
    "L13": ("Frankfurt","Hannover"), "L14": ("Frankfurt","Koeln"), "L15": ("Frankfurt","Leipzig"),
    "L16": ("Frankfurt","Mannheim"), "L17": ("Frankfurt","Nuernberg"), "L18": ("Hamburg","Hannover"),
    "L19": ("Hannover","Leipzig"), "L20": ("Karlsruhe","Mannheim"), "L21": ("Karlsruhe","Stuttgart"),
    "L22": ("Leipzig","Nuernberg"), "L23": ("Muenchen","Nuernberg"), "L24": ("Muenchen","Ulm"),
    "L25": ("Nuernberg","Stuttgart"), "L26": ("Stuttgart","Ulm"),
}
nodes = list(raw_nodes.keys())
def key(e):
    return tuple(sorted(e))
links = [key(v) for v in raw_links.values()]
print(f"{len(nodes)} nodes, {len(links)} links")

In [ ]:
# ----------------------------- Configuration -----------------------------
CONFIG = dict(
    epochs      = 1000,          # training epochs per (scenario, seed)
    lr          = 0.001,
    hold_frac   = 0.4,           # fraction of entities withheld from the loss (stratified)
    seeds       = [0, 1, 2, 3, 4],
    scenarios   = None,          # None = all 26 links; or e.g. ["L1","L8","L13"]
    # Optional real SNDlib demand matrix. If the file is missing we fall back to
    # all-pairs unit demands (still a valid reroutability test on the same topology).
    demand_file = "",            # e.g. ".../demandMatrix-germany17-DFN-5min-20050215-0000.txt"
    quick_test  = False,         # True -> 3 scenarios x 2 seeds x 200 epochs (smoke test)
)
if CONFIG["quick_test"]:
    CONFIG.update(epochs=200, seeds=[0,1], scenarios=["L1","L8","L13"])
CONFIG

In [ ]:
# ----------------------------- Demand loader (real file or fallback) -----------------------------
def parse_sndlib_demands(filepath):
    sources, targets, values = [], [], []
    with open(filepath) as f:
        in_section = False
        for line in f:
            s = line.strip()
            if s == "DEMANDS (":
                in_section = True; continue
            if s == ")":
                in_section = False; continue
            if in_section and s:
                p = s.split()
                if len(p) >= 7:
                    sources.append(p[2]); targets.append(p[3]); values.append(float(p[6]))
    return list(zip(sources, targets, values))

def load_demands(cfg):
    if cfg["demand_file"] and os.path.exists(cfg["demand_file"]):
        raw = parse_sndlib_demands(cfg["demand_file"])
        demands = [(s, t) for (s, t, v) in raw if s in raw_nodes and t in raw_nodes and v > 0]
        print(f"Loaded {len(demands)} demands from SNDlib file.")
    else:
        demands = list(itertools.combinations(nodes, 2))   # all-pairs unit demands
        print(f"Demand file not set/found -> fallback: {len(demands)} all-pairs unit demands.")
    return demands

DEMANDS = load_demands(CONFIG)

In [ ]:
# ----------------------------- Ground-truth generators -----------------------------
def gt_topological(down_link):
    """1-hop failure spreading (paper assumption). affected = down link's endpoints and
    every link incident to them."""
    u, v = down_link
    aff_nodes = {u, v}
    aff_links = {l for l in links if l[0] in (u, v) or l[1] in (u, v)}
    return ({n: int(n in aff_nodes) for n in nodes},
            {l: int(l in aff_links) for l in links})

def _sp(G, s, t):
    return nx.shortest_path(G, s, t) if (s in G and t in G and nx.has_path(G, s, t)) else None

def gt_demand_footprint(down_link, demands):
    """Reroute footprint: only demands whose baseline shortest path used the failed edge are
    rerouted; affected links/nodes are the *new* edges/nodes their traffic moves onto (plus the
    failed edge and its endpoints). Independent of the injected topological rule."""
    G = nx.Graph(); G.add_nodes_from(nodes); G.add_edges_from(links)
    de = key(down_link)
    base_paths = {}
    direct = []
    for i, (s, t) in enumerate(demands):
        p = _sp(G, s, t)
        if p is None:
            continue
        base_paths[i] = p
        if de in {key((a, b)) for a, b in zip(p, p[1:])}:
            direct.append(i)
    G2 = G.copy(); G2.remove_edge(*down_link)
    aff_links = {de}; aff_nodes = set(down_link)
    for i in direct:
        s, t = demands[i]
        np_ = _sp(G2, s, t)
        if np_ is None:                      # demand dropped -> its endpoints are impacted
            aff_nodes.update([s, t]); continue
        old_e = {key((a, b)) for a, b in zip(base_paths[i], base_paths[i][1:])}
        old_n = set(base_paths[i])
        for a, b in zip(np_, np_[1:]):
            k = key((a, b))
            if k not in old_e: aff_links.add(k)
        for n in np_:
            if n not in old_n: aff_nodes.add(n)
    return ({n: int(n in aff_nodes) for n in nodes},
            {l: int(l in aff_links) for l in links})

# quick check
gtn_t, gtl_t = gt_topological(key(("Berlin","Hamburg")))
gtn_d, gtl_d = gt_demand_footprint(key(("Berlin","Hamburg")), DEMANDS)
print("Berlin-Hamburg  topo affected: %d links, %d nodes" % (sum(gtl_t.values()), sum(gtn_t.values())))
print("Berlin-Hamburg  demand affected: %d links, %d nodes" % (sum(gtl_d.values()), sum(gtn_d.values())))

In [ ]:
# ----------------------------- LTN predicate model & operators -----------------------------
# Predicate MLP (sigmoid output), identical to the case-study notebook (knowledge_embedding.ipynb).
class PredMLP(torch.nn.Module):
    def __init__(self, layer_sizes=(4, 32, 32, 1)):
        super().__init__()
        self.elu = torch.nn.ELU()
        self.sigmoid = torch.nn.Sigmoid()
        self.linear_layers = torch.nn.ModuleList(
            [torch.nn.Linear(layer_sizes[i-1], layer_sizes[i]) for i in range(1, len(layer_sizes))])
        self.to(device)
    def forward(self, *x):
        x = list(x)
        x = x[0] if len(x) == 1 else torch.cat(x, dim=1)
        x = x.to(device)
        for layer in self.linear_layers[:-1]:
            x = self.elu(layer(x))
        return self.sigmoid(self.linear_layers[-1](x))

Not     = ltn.Connective(ltn.fuzzy_ops.NotStandard())
Implies = ltn.Connective(ltn.fuzzy_ops.ImpliesReichenbach())
Forall  = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")
SatAgg  = ltn.fuzzy_ops.SatAgg()

In [ ]:
# ----------------------------- Stratified known / held-out split -----------------------------
def make_split(gt_node, gt_link, down_link, seed, hold_frac):
    """Hold out a stratified fraction of nodes and links (>=1 affected and >=1 healthy when
    possible). The failed edge is never in the pool; its down-fact is always given as the event."""
    rng = np.random.default_rng(seed)
    de = key(down_link)
    link_pool = [l for l in links if l != de]
    node_pool = list(nodes)
    def strat(pool, lab):
        pos = [x for x in pool if lab[x]]; neg = [x for x in pool if not lab[x]]
        rng.shuffle(pos); rng.shuffle(neg)
        hp = max(1, int(round(hold_frac*len(pos)))) if pos else 0
        hn = max(1, int(round(hold_frac*len(neg)))) if neg else 0
        return set(pos[:hp]) | set(neg[:hn])
    held_links = strat(link_pool, gt_link)
    held_nodes = strat(node_pool, gt_node)
    known_links = [l for l in link_pool if l not in held_links]
    known_nodes = [n for n in node_pool if n not in held_nodes]
    return known_nodes, known_links, held_nodes, held_links

In [ ]:
# ----------------------------- Train one scenario (known-only facts + rules) -----------------------------
def train_scenario(down_link, gt_node, gt_link, known_nodes, known_links, seed, cfg):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    # Fixed coordinate embeddings (non-trainable), as in the case study -> the MLP generalises
    # spatially, which is what makes held-out prediction a genuine test.
    nc = {k: ltn.Constant(torch.tensor(v, dtype=torch.float32, device=device)) for k, v in raw_nodes.items()}
    Link     = ltn.Predicate(PredMLP((4, 32, 32, 1)))
    GoodNode = ltn.Predicate(PredMLP((2, 32, 32, 1)))
    BadNode  = ltn.Predicate(PredMLP((2, 32, 32, 1)))
    n_ = ltn.Variable("n", torch.stack([c.value for c in nc.values()]))
    m_ = ltn.Variable("m", torch.stack([c.value for c in nc.values()]))
    params = list(Link.parameters()) + list(GoodNode.parameters()) + list(BadNode.parameters())
    opt = torch.optim.Adam(params, lr=cfg["lr"])
    u, v = down_link
    for epoch in range(cfg["epochs"]):
        opt.zero_grad()
        facts = [Not(Link(nc[u], nc[v]))]                       # the event: failed link is down
        for (s, t) in known_links:                              # known link grounding facts
            facts.append(Not(Link(nc[s], nc[t])) if gt_link[(s, t)] else Link(nc[s], nc[t]))
        for nm in known_nodes:                                  # known node grounding facts
            facts.append(BadNode(nc[nm]) if gt_node[nm] else GoodNode(nc[nm]))
        sat = SatAgg(
            *facts,
            Forall(n_, Not(Link(n_, n_)), p=5),                 # anti-reflexive
            Forall([n_, m_], Implies(Link(n_, m_), Link(m_, n_)), p=10),  # symmetric
            Forall(n_, Implies(BadNode(n_), Not(GoodNode(n_))), p=5),
        )
        loss = 1.0 - sat
        loss.backward(); opt.step()
    return Link, GoodNode, BadNode, nc

In [ ]:
# ----------------------------- Evaluate on held-out entities -----------------------------
def _metrics(y_true, y_score):
    y_true = np.asarray(y_true); y_score = np.asarray(y_score)
    out = dict(n=len(y_true), n_pos=int(y_true.sum()))
    if len(y_true) == 0:
        return {**out, "acc": np.nan, "prec": np.nan, "rec": np.nan, "f1": np.nan, "auroc": np.nan}
    y_pred = (y_score > 0.5).astype(int)
    out["acc"] = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    out.update(prec=p, rec=r, f1=f1)
    out["auroc"] = roc_auc_score(y_true, y_score) if len(set(y_true)) > 1 else np.nan
    return out

def evaluate(Link, GoodNode, BadNode, nc, held_nodes, held_links, gt_node, gt_link):
    yl_true, yl_score = [], []
    for (s, t) in held_links:
        up = 0.5 * (Link(nc[s], nc[t]).value.item() + Link(nc[t], nc[s]).value.item())
        yl_score.append(1.0 - up)                 # affected/down score
        yl_true.append(gt_link[(s, t)])
    yn_true, yn_score = [], []
    for nm in held_nodes:
        yn_score.append(BadNode(nc[nm]).value.item())
        yn_true.append(gt_node[nm])
    return _metrics(yl_true, yl_score), _metrics(yn_true, yn_score)

In [ ]:
# ----------------------------- Main sweep: scenarios x seeds -----------------------------
scen_names = CONFIG["scenarios"] or list(raw_links.keys())
rows = []
for si, lname in enumerate(scen_names):
    down = key(raw_links[lname])
    gtn_t, gtl_t = gt_topological(down)                          # training facts + topo eval
    gtn_d, gtl_d = gt_demand_footprint(down, DEMANDS)            # independent demand eval
    for seed in CONFIG["seeds"]:
        kn_n, kn_l, hd_n, hd_l = make_split(gtn_t, gtl_t, down, seed, CONFIG["hold_frac"])
        Link, GoodNode, BadNode, nc = train_scenario(down, gtn_t, gtl_t, kn_n, kn_l, seed, CONFIG)
        for gt_name, (gn, gl) in [("topological", (gtn_t, gtl_t)), ("demand", (gtn_d, gtl_d))]:
            m_link, m_node = evaluate(Link, GoodNode, BadNode, nc, hd_n, hd_l, gn, gl)
            for ent, m in [("link", m_link), ("node", m_node)]:
                rows.append(dict(scenario=lname, seed=seed, gt=gt_name, entity=ent, **m))
    print(f"[{si+1}/{len(scen_names)}] {lname} done")
results = pd.DataFrame(rows)
print("collected", len(results), "rows")
results.head()

In [ ]:
# ----------------------------- Aggregate -> mean +/- std -----------------------------
agg = (results
       .groupby(["gt", "entity"])[["acc", "prec", "rec", "f1", "auroc"]]
       .agg(["mean", "std"]))
pd.set_option("display.float_format", lambda x: f"{x:.3f}")
print(agg)

# Compact summary table (mean +/- std across scenarios x seeds)
def fmt(g, e, col):
    sub = results[(results.gt == g) & (results.entity == e)][col].dropna()
    return f"{sub.mean():.3f} +/- {sub.std():.3f}"
summary = pd.DataFrame([
    dict(GroundTruth=g, Entity=e,
         Accuracy=fmt(g, e, "acc"), F1=fmt(g, e, "f1"), AUROC=fmt(g, e, "auroc"))
    for g in ["topological", "demand"] for e in ["node", "link"]
])
summary

In [ ]:
# ----------------------------- Save outputs -----------------------------
outdir = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else "."
results.to_csv(os.path.join(outdir, "heldout_results_raw.csv"), index=False)
summary.to_csv(os.path.join(outdir, "heldout_summary.csv"), index=False)
print("Saved heldout_results_raw.csv and heldout_summary.csv")

## Notes for the write-up

* **What the numbers mean.** Each metric is computed on **held-out** entities only, averaged over
  all link-down scenarios x seeds (mean +/- std). The *topological* rows measure whether the model
  faithfully **completes** the injected knowledge over the unknown region; the *demand* rows measure
  whether that reasoning also predicts **real traffic impact** (an independent semantic).
* **AUROC is the headline metric** — it needs no threshold, so it avoids the arbitrary mid-range
  cutoff the reviewers criticised.
* **Non-circularity.** Training facts use only the *known* entities' topological labels; the *demand*
  ground truth is never injected. Held-out labels are never in the loss.
* **Reproducing Fig. 13 quantitatively.** The Berlin-Hamburg scenario corresponds to `L1`; its held-out
  metrics are the quantitative analogue of the two-panel figure.
* **Runtime.** ~`len(scenarios) x len(seeds)` training runs. Use `quick_test=True` first. GPU recommended
  for the full 26 x 5 sweep.
